**Assignment 6 - Design and implement a Convolutional Neural Network (CNN) for image classification using the Tomato or Soybean disease dataset.**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yusufmurtaza01/tomato-leaf-dataset-for-disease-detection")

print("Path to dataset files:", path)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os
import kagglehub

# Re-download/retrieve path to ensure variable is in scope
path = kagglehub.dataset_download("yusufmurtaza01/tomato-leaf-dataset-for-disease-detection")

# Configuration
BATCH_SIZE = 32
IMG_SIZE = (128, 128)

# The previous output shows a 'tomato_yolo_dataset' folder.
# Let's adjust the path to look inside that subfolder if necessary.
adjusted_path = os.path.join(path, 'tomato_yolo_dataset')
if not os.path.exists(adjusted_path):
    adjusted_path = path

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    adjusted_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    adjusted_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Detected classes: {class_names}")

100%|██████████| 280M/280M [00:03<00:00, 77.8MB/s]

Extracting files...


Found 18158 files belonging to 2 classes.
Using 14527 files for training.


KeyboardInterrupt: 

In [ ]:
# Optimize for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Build CNN Model
num_classes = len(class_names)
# If num_classes is 1, we use sigmoid; otherwise softmax.
activation_fn = 'sigmoid' if num_classes == 1 else 'softmax'
loss_fn = 'binary_crossentropy' if num_classes == 1 else 'sparse_categorical_crossentropy'

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation=activation_fn)
])

model.compile(
    optimizer='adam',
    loss=loss_fn,
    metrics=['accuracy']
)

model.summary()

NameError: name 'train_ds' is not defined

In [ ]:
# Train the model
EPOCHS = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Plot results
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


 97/454 ━━━━━━━━━━━━━━━━━━━━ 5:15 884ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00